# 06 · The `condition_occurrence` table

Diagnoses, problems, signs and symptoms the patient **has**. The predefined
`condition_occurrence` definition follows the CDM conventions, so negated, ruled-out,
past and family conditions are *not* rows here.

Each condition gets a required `status` label, stored in `condition_status_concept_id`:

| Label | Means | Concept |
|---|---|---|
| `confirmed` | stated as a fact | 32893 *Confirmed diagnosis* |
| `preliminary` | a likely working diagnosis ("probable", "presumed") | 32899 *Preliminary diagnosis* |

Details: [the condition_occurrence table](https://precisiondata.github.io/aatm/getting-started/omop_tables/#the-condition_occurrence-table).

In [1]:
# Run from the folder where you ran `aatm init` (it holds .aatm/ and .env).
import os

if not os.path.isdir(".aatm") and os.path.isdir("../.aatm"):
    os.chdir("..")

## 1. Mapper and notes

The same mapper as in notebook 04 codes the extracted names.

In [2]:
from aatm.omop import ClinicalNote, OmopExtractionPipeline
from aatm.selectors import OpenAILLMSelector
from aatm.terminology_mapper import TerminologyMapper

mapper = TerminologyMapper(
    retriever="text-embedding-3-small", selector=OpenAILLMSelector("gpt-5-mini")
)
mapper.retriever.top_k = 30

notes = [
    ClinicalNote(
        text=(
            "Type 2 diabetes since 2019. Probable community-acquired pneumonia. "
            "History of appendicitis. Rule out PE. Denies chest pain. Mother had breast cancer."
        ),
        person_id=20,
        note_date="2025-06-01",
    )
]

## 2. Run

In [3]:
pipeline = OmopExtractionPipeline(
    "condition_occurrence", model_id="openai/gpt-5-mini", mapper=mapper
)
tables = await pipeline(notes)

tables["condition_occurrence"][
    [
        "condition_source_value",
        "condition_concept_id",
        "condition_start_date",
        "condition_end_date",
        "condition_status_concept_id",
        "condition_status_source_value",
    ]
]

,condition_source_value,condition_concept_id,condition_start_date,condition_end_date,condition_status_concept_id,condition_status_source_value
0,Type 2 diabetes,201826,2019-01-01,None,32893,confirmed
1,community-acquired pneumonia,4293463,2025-06-01,None,32899,preliminary


What is **not** a row, and where it goes instead:

| In the note | Why |
|---|---|
| "History of appendicitis" | past diagnosis → `observation`: add `medical_history` (notebook 05) |
| "Rule out PE" | rule-out diagnoses are not recorded |
| "Denies chest pain" | negated |
| "Mother had breast cancer" | family history → `observation`: add `family_history` |

**Dates:** `condition_start_date` is the date the note gives ("since 2019" → 2019-01-01,
reported as `imputed_date_parts`), else the note date, so read it as "recorded on" rather
than "onset". `condition_end_date` is filled only when the note gives one.

## 3. Check the coded concepts and the issues

`condition_concept_id` only accepts standard Condition concepts; a concept of another domain is stored but reported as `domain_mismatch`.

In [4]:
tables.mappings_dataframe()[
    ["row_id", "column", "text", "concept_id", "concept_name", "domain_id", "score"]
]

,row_id,column,text,concept_id,concept_name,domain_id,score
0,1,condition_concept_id,Type 2 diabetes,201826,Type 2 diabetes mellitus,Condition,0.853669
1,2,condition_concept_id,community-acquired pneumonia,4293463,Community acquired pneumonia,Condition,0.778075


In [5]:
tables.issues_dataframe()[["row_id", "column", "reason", "detail"]]

,row_id,column,reason,detail
0,1,condition_start_date,imputed_date_parts,'2019' has no day or month; read as 2019-01-01


## Next

Medicines, vaccines and biologics: [07_drug_exposure.ipynb](07_drug_exposure.ipynb).